<a href="https://colab.research.google.com/github/EvansAmarh/CausalMedia-GH/blob/main/Trained_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import files
uploaded = files.upload()

Saving synthetic_school_data.csv to synthetic_school_data.csv


In [3]:
from google.colab import files
uploaded = files.upload()

Saving student_level_dataset.csv to student_level_dataset.csv


In [4]:
df_ednet = pd.read_csv('student_level_dataset.csv')
df_synth = pd.read_csv('synthetic_school_data.csv')

print("EdNet shape:", df_ednet.shape)
print("EdNet unique students:", df_ednet['student_id'].nunique())

print("\nSynthetic shape:", df_synth.shape)
print("Synthetic unique students:", df_synth['student_id'].nunique())

EdNet shape: (10000, 10)
EdNet unique students: 10000

Synthetic shape: (50000, 6)
Synthetic unique students: 50000


In [5]:

df_ednet = pd.read_csv('student_level_dataset.csv')
df_synth  = pd.read_csv('synthetic_school_data.csv')
df_synth_small = df_synth.sample(n=len(df_ednet), random_state=42).reset_index(drop=True)
df_synth_small['student_id'] = df_ednet['student_id'].values
df = pd.merge(df_ednet, df_synth_small, on='student_id', how='inner')
df['peer_activity_index'] = df.groupby('location')['total_interactions'] \
                               .rank(pct=True)

if 'content_modality_preference' not in df.columns:
    np.random.seed(42)
    df['content_modality_preference'] = np.random.beta(2, 3, size=len(df))

print("Final shape:", df.shape)
print("Unique students:", df['student_id'].nunique())
print("\nMissing values:\n", df.isnull().sum())
print("\nSample:\n", df.head(3))

Final shape: (10000, 17)
Unique students: 10000

Missing values:
 student_id                     0
multimedia_engagement          0
performance_gain               0
prior_achievement              0
consistency                    0
early_struggle                 0
skill_coverage                 0
session_duration_avg           0
total_interactions             0
peer_collaboration             0
location                       0
tablet_access                  0
bandwidth_category             0
school_resource_level          0
teacher_qual                   0
peer_activity_index            0
content_modality_preference    0
dtype: int64

Sample:
   student_id  multimedia_engagement  performance_gain  prior_achievement  \
0    u100001               0.454545         -0.272727           0.727273   
1    u100092               0.488372          0.033333           0.500000   
2    u100194               0.500000         -0.200000           0.600000   

   consistency  early_struggle  skill_coverag

In [6]:
df.to_csv('final_dataset.csv', index=False)
print("Saved. Ready for Role 4 modelling.")

T = df['multimedia_engagement'].values
Y = df['performance_gain'].values

confounder_cols = [
    'prior_achievement', 'content_modality_preference',
    'bandwidth_category', 'consistency', 'early_struggle',
    'skill_coverage', 'session_duration_avg', 'peer_activity_index',
    'school_resource_level', 'tablet_access', 'teacher_qual',
    'peer_collaboration'
]
X = df[confounder_cols].values

print("T shape:", T.shape)
print("Y shape:", Y.shape)
print("X shape:", X.shape)

Saved. Ready for Role 4 modelling.
T shape: (10000,)
Y shape: (10000,)
X shape: (10000, 12)


In [7]:
!pip install econml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 114.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 68.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 17.5 MB/s eta 0:00:00
  Attempting uninstall: shap
    Found existing installation: shap 0.52.0
    Uninstalling shap-0.52.0:
      Successfully uninstalled shap-0.52.0


In [9]:
from scipy import stats
from sklearn.linear_model import LinearRegression
from econml.dml import LinearDML
from lightgbm import LGBMRegressor
import numpy as np

# BASELINE B1: Naive Pearson Correlation
pearson_r, pearson_p = stats.pearsonr(T, Y)
print(f"B1 Pearson r: {pearson_r:.4f}, p-value: {pearson_p:.4f}")

# BASELINE B2: OLS with confounders
X_with_T = np.column_stack([T, X])
ols = LinearRegression().fit(X_with_T, Y)
ols_coef = ols.coef_[0]
print(f"B2 OLS coefficient on T: {ols_coef:.4f}")

# BASELINE B3: Linear DML
linear_dml = LinearDML(
    model_y=LGBMRegressor(n_estimators=100, random_state=42),
    model_t=LGBMRegressor(n_estimators=100, random_state=42),
    random_state=42
)
linear_dml.fit(Y, T, X=X)
linear_dml_ate = linear_dml.ate(X)
print(f"B3 Linear DML ATE: {linear_dml_ate:.4f}")

B1 Pearson r: 0.0500, p-value: 0.0000
B2 OLS coefficient on T: 0.0949


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


B3 Linear DML ATE: 0.0665


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/econml/sklearn_extensions/linear_model.py:1815: UserWarning: Co-variance matrix is underdetermined. Inference will be invalid!
  warnings.warn("Co-variance matrix is underdetermined. Inference will be inv

In [10]:
from econml.dml import CausalForestDML
from lightgbm import LGBMRegressor
import pandas as pd
X_df = pd.DataFrame(X, columns=confounder_cols)

# SEED 42
print("Training seed 42...")
cf_42 = CausalForestDML(
    model_y=LGBMRegressor(n_estimators=200, random_state=42, verbose=-1),
    model_t=LGBMRegressor(n_estimators=200, random_state=42, verbose=-1),
    n_estimators=1000, honest=True, min_samples_leaf=5,
    random_state=42, discrete_treatment=False
)
cf_42.fit(Y, T, X=X_df)
ate_42 = cf_42.ate(X_df)
print(f"Seed 42 ATE: {ate_42:.4f}")

# SEED 123
print("Training seed 123...")
cf_123 = CausalForestDML(
    model_y=LGBMRegressor(n_estimators=200, random_state=123, verbose=-1),
    model_t=LGBMRegressor(n_estimators=200, random_state=123, verbose=-1),
    n_estimators=1000, honest=True, min_samples_leaf=5,
    random_state=123, discrete_treatment=False
)
cf_123.fit(Y, T, X=X_df)
ate_123 = cf_123.ate(X_df)
print(f"Seed 123 ATE: {ate_123:.4f}")

# SEED 777
print("Training seed 777...")
cf_777 = CausalForestDML(
    model_y=LGBMRegressor(n_estimators=200, random_state=777, verbose=-1),
    model_t=LGBMRegressor(n_estimators=200, random_state=777, verbose=-1),
    n_estimators=1000, honest=True, min_samples_leaf=5,
    random_state=777, discrete_treatment=False
)
cf_777.fit(Y, T, X=X_df)
ate_777 = cf_777.ate(X_df)
print(f"Seed 777 ATE: {ate_777:.4f}")

# SUMMARY
ate_mean = np.mean([ate_42, ate_123, ate_777])
ate_std  = np.std([ate_42, ate_123, ate_777])
print(f"\nFinal ATE: {ate_mean:.4f} ± {ate_std:.4f}")
print(f"Inflation ratio (B1/ATE): {pearson_r/ate_mean:.2f}x")

Training seed 42...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

Seed 42 ATE: 0.0396
Training seed 123...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

Seed 123 ATE: 0.0523
Training seed 777...


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/v

Seed 777 ATE: 0.0258

Final ATE: 0.0392 ± 0.0108
Inflation ratio (B1/ATE): 1.28x


In [12]:
# CONFIDENCE INTERVAL (seed 42 as primary)
ci = cf_42.ate_interval(X_df, alpha=0.05)
print(f"95% CI: [{ci[0]:.4f}, {ci[1]:.4f}]")

if ci[0] > 0:
    print("PASS — CI excludes zero. Effect is statistically significant.")
elif ci[1] < 0:
    print("PASS — CI excludes zero (negative effect).")
else:
    print("NOTE — CI includes zero. Effect is not statistically significant.")

# CATE (individual treatment effects)
cate = cf_42.effect(X_df)
df['cate'] = cate

print(f"\nCATE mean:  {cate.mean():.4f}")
print(f"CATE std:   {cate.std():.4f}")
print(f"CATE min:   {cate.min():.4f}")
print(f"CATE max:   {cate.max():.4f}")

# SUBGROUP ANALYSIS
print("\n=== CATE by Bandwidth Category ===")
print(df.groupby('bandwidth_category')['cate'].mean().round(4))

print("\n=== CATE by Location ===")
print(df.groupby('location')['cate'].mean().round(4))

print("\n=== CATE by Achievement Quartile ===")
df['achievement_q'] = pd.qcut(df['prior_achievement'], q=4,
                               labels=['Q1-Low','Q2','Q3','Q4-High'])
print(df.groupby('achievement_q', observed=False)['cate'].mean().round(4))

# SAVE OUTPUTS
df.to_csv('results_with_cate.csv', index=False)

import pickle
with open('cf_model_42.pkl', 'wb') as f:
    pickle.dump(cf_42, f)

import json
summary = {
    'pearson_r': round(pearson_r, 4),
    'ols_coef': round(float(ols_coef), 4),
    'linear_dml_ate': round(float(linear_dml_ate), 4),
    'ate_seed42': round(float(ate_42), 4),
    'ate_seed123': round(float(ate_123), 4),
    'ate_seed777': round(float(ate_777), 4),
    'ate_mean': round(float(ate_mean), 4),
    'ate_std': round(float(ate_std), 4),
    'ci_lower': round(float(ci[0]), 4),
    'ci_upper': round(float(ci[1]), 4),
    'inflation_ratio': round(float(pearson_r / ate_mean), 4)
}
with open('model_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)



95% CI: [-0.6026, 0.6817]
NOTE — CI includes zero. Effect is not statistically significant.

CATE mean:  0.0396
CATE std:   0.2668
CATE min:   -1.1256
CATE max:   1.4067

=== CATE by Bandwidth Category ===
bandwidth_category
1    0.1263
2   -0.0053
3    0.0177
Name: cate, dtype: float64

=== CATE by Location ===
location
Peri-urban   -0.0015
Rural         0.1238
Urban         0.0172
Name: cate, dtype: float64

=== CATE by Achievement Quartile ===
achievement_q
Q1-Low     0.0548
Q2        -0.0041
Q3         0.0309
Q4-High    0.0755
Name: cate, dtype: float64


In [13]:
import json, pickle

# Save dataset with CATE
df.to_csv('results_with_cate.csv', index=False)

# Save model
with open('cf_model_42.pkl', 'wb') as f:
    pickle.dump(cf_42, f)

# Save summary JSON
summary = {
    'pearson_r':        round(float(pearson_r), 4),
    'ols_coef':         round(float(ols_coef), 4),
    'linear_dml_ate':   round(float(linear_dml_ate), 4),
    'ate_seed42':       round(float(ate_42), 4),
    'ate_seed123':      round(float(ate_123), 4),
    'ate_seed777':      round(float(ate_777), 4),
    'ate_mean':         round(float(ate_mean), 4),
    'ate_std':          round(float(ate_std), 4),
    'ci_lower':         round(float(ci[0]), 4),
    'ci_upper':         round(float(ci[1]), 4),
    'cate_mean':        round(float(df['cate'].mean()), 4),
    'cate_std':         round(float(df['cate'].std()), 4),
    'inflation_ratio':  round(float(pearson_r / ate_mean), 4)
}
with open('model_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Saved: results_with_cate.csv")
print("Saved: cf_model_42.pkl")
print("Saved: model_summary.json")
print("\nSummary:")
for k, v in summary.items():
    print(f"  {k}: {v}")

Saved: results_with_cate.csv
Saved: cf_model_42.pkl
Saved: model_summary.json

Summary:
  pearson_r: 0.05
  ols_coef: 0.0949
  linear_dml_ate: 0.0665
  ate_seed42: 0.0396
  ate_seed123: 0.0523
  ate_seed777: 0.0258
  ate_mean: 0.0392
  ate_std: 0.0108
  ci_lower: -0.6026
  ci_upper: 0.6817
  cate_mean: 0.0396
  cate_std: 0.2669
  inflation_ratio: 1.2755


In [14]:
import shutil, os

save_dir = '/content/drive/MyDrive/Datasets/'
os.makedirs(save_dir, exist_ok=True)

for fname in ['results_with_cate.csv', 'cf_model_42.pkl', 'model_summary.json']:
    shutil.copy(fname, save_dir + fname)
    print(f"Backed up: {fname}")

Backed up: results_with_cate.csv
Backed up: cf_model_42.pkl
Backed up: model_summary.json
